# NVIDIA: THE FALL — The CEO's Last Chance
A **fictional** cyber-forensics escape room. All people, accounts, devices and events are invented.

| | Version 1 | Version 2 |
|---|---|---|
| Stack | Streamlit | FastAPI + HTML/CSS/JS |
| Idea | Type to a Gemini Game Master in one room; Python rules decide | Cinematic point-and-click game, three rooms; Python engine validates every action |
| Section | cell 5 | cell 6 |

Run cells 1-4 **in order, once**. Then start Version 1 and/or Version 2 independently (cells 5 and 6); they can run side by side.

**Gemini key (optional):** click the key icon in Colab's left sidebar, add a secret named `GEMINI_API_KEY`, and switch on *Notebook access*. The key is never typed into code or files. Without it, both apps use the built-in offline narrator and are fully playable.

### 1. Upload the project ZIP
Choose `NVIDIA_The_Fall_rebuild.zip` in the file dialog. If a ZIP is already in the Colab file panel, the upload is skipped.

In [ ]:
import os, glob
os.makedirs("/content", exist_ok=True)
os.chdir("/content")
ZIP_NAME = "NVIDIA_The_Fall_rebuild.zip"
if not glob.glob("/content/*.zip"):
    try:
        from google.colab import files
        files.upload()                       # opens the file chooser
    except ImportError:
        print("Not running in Colab: place", ZIP_NAME, "in /content and continue.")
zips = glob.glob(f"/content/{ZIP_NAME}") or glob.glob("/content/*.zip")
print("ZIP found:" if zips else "NO ZIP FOUND. Re-run this cell and upload " + ZIP_NAME)
for z in zips: print(" ", z)

### 2. Extract and locate the project folder

In [ ]:
import sys, zipfile
from pathlib import Path
if not zips:
    raise SystemExit("Upload the ZIP first (cell 1).")
zpath = Path(zips[0])
if not zipfile.is_zipfile(zpath):
    raise SystemExit(f"{zpath.name} is not a valid ZIP. Upload the file again.")
with zipfile.ZipFile(zpath) as z:
    bad = [n for n in z.namelist() if n.startswith("/") or ".." in Path(n).parts]
    if bad:
        raise SystemExit(f"Unsafe paths in ZIP: {bad[:3]}")
    z.extractall("/content")
found = list(Path("/content").glob("*/colab/launch.py"))
if not found:
    raise SystemExit("colab/launch.py not found after extracting. This is not the NVIDIA_The_Fall_rebuild.zip project.")
sys.path.insert(0, str(found[0].parent.parent))
from colab import launch as L
try:
    PROJECT = L.find_project("/content")
except FileNotFoundError as e:
    raise SystemExit(str(e))
os.chdir(PROJECT)
print("Project folder:", PROJECT)
print("Contents:", ", ".join(sorted(p.name for p in PROJECT.iterdir() if not p.name.startswith("."))))

### 3. Install requirements and configure Gemini

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-1500:], r.stderr[-1500:])
    raise SystemExit("pip install failed. See the message above and re-run this cell.")
print("Requirements installed.")
HAS_KEY = L.load_key()   # reads Colab Secret GEMINI_API_KEY; never prints it
print("Gemini key:", "found (Gemini narrator ON)" if HAS_KEY else "not found -> OFFLINE mode (add the secret named GEMINI_API_KEY to enable Gemini)")

### 4. Run the automated tests
The tests never call Gemini and need no key. The last lines show the exact passed / failed / skipped counts.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rs", "-p", "no:cacheprovider"], cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-3500:], r.stderr[-800:])
print("TESTS PASSED" if r.returncode == 0 else f"TESTS FAILED (exit code {r.returncode}): fix before launching.")

### 5. Version 1: Streamlit Game Master
Starts the app, prints its URL and embeds it below. If port 8501 is busy it uses the next free port.
**To stop it:** run the stop cell right below (`v1.stop()`), or use *Runtime ▸ Restart session*. Starting it again after a stop is safe.

In [ ]:
try:
    v1 = L.launch_streamlit(PROJECT, port=8501)
    print("Stop with:  v1.stop()")
except Exception as e:
    print("Version 1 failed to start:\n", e)

In [ ]:
# Stop Version 1 (run this cell when you are done):
try:
    v1.stop()
except NameError:
    print("Version 1 is not running.")

### 6. Version 2: FastAPI cinematic game
Starts the FastAPI server (port 8000, or the next free one), prints its URL and embeds the game.
**To stop it:** run the stop cell below (`v2.stop()`). Version 1 keeps running if it is open.

In [ ]:
try:
    v2 = L.launch_fastapi(PROJECT, port=8000)
    print("API health check:", v2.url.rstrip("/") + "/api/health")
    print("Stop with:  v2.stop()")
except Exception as e:
    print("Version 2 failed to start:\n", e)

In [ ]:
# Stop Version 2 (run this cell when you are done):
try:
    v2.stop()
except NameError:
    print("Version 2 is not running.")

### Troubleshooting
- **No ZIP found / not a valid ZIP:** re-run cell 1 and upload `NVIDIA_The_Fall_rebuild.zip` again.
- **Port in use:** the launcher moves to the next free port and prints the URL it actually used.
- **Startup failure:** the error shows the last lines of the app's log (`.version1_streamlit.log` / `.version2_fastapi.log` in the project folder).
- **Blank frame:** open the printed URL in a new browser tab.
- **No Gemini key:** the offline narrator is used; both games stay fully playable and the AI never gives the answers.
- Everything is fictional; account activity in the story is never proof of who a person is.